## Notebook configuration & run mode ⚙️

Edit the *configuration code cell immediately below* to set runtime parameters used throughout this notebook (for example: `MODEL_NAME`, `MODEL_VERSION`, `RUN_MODE` ('train' or 'inference'), `TARGET_SIZE`, `BATCH_SIZE`, `NUM_EPOCHS`, `PATIENCE`, and `CHECKPOINT_PATH`). After changing values, re-run that configuration cell to apply them.

Tip: Run the **Imports & environment** cell after updating configuration so dependent cells pick up the new settings. Keep the `CHECKPOINT_PATH` empty when training from scratch, or set it to a saved checkpoint for inference/finetuning.

In [ ]:
## Notebook configuration & run mode ⚙️
Edit this cell to set runtime parameters used throughout the notebook (model version, training vs inference mode, epochs, target size, etc). After changing values, re-run the cell to apply them.


In [ ]:
# Top-level parameters (edit and re-run this cell)
MODEL_NAME = 'boundary_regressor'
MODEL_VERSION = 'v0.1'
RUN_MODE = 'train'  # 'train' or 'inference'
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

# Training hyperparams
TARGET_SIZE = (256, 256)
BATCH_SIZE = 4
NUM_EPOCHS = 12
PATIENCE = 3
LEARNING_RATE = 1e-3

# Misc
SAVE_DIR = Path(os.environ.get('PARKING_SAVE_DIR', './experiments'))
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint path (set when you want to load a specific checkpoint for inference)
CHECKPOINT_PATH = None  # e.g., './experiments/best.pth'

print('Configuration loaded: MODEL_NAME=', MODEL_NAME, 'MODEL_VERSION=', MODEL_VERSION, 'RUN_MODE=', RUN_MODE)


In [ ]:
# 1. Import required libraries
import os
import json
from pathlib import Path
import glob
import random
import math

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Optional: albumentations will be used in training notebooks for synchronized augmentations
try:
    import albumentations as A
except Exception:
    A = None

print('OpenCV version', cv2.__version__)
print('PyTorch version', torch.__version__)


In [ ]:
### Imports & environment
This cell imports core libraries and prints runtime versions. Run it once after editing the top-level configuration if you change the `DEVICE` or package requirements.


In [ ]:
# Colab setup (run this cell in Colab)
try:
    import google.colab
    from google.colab import drive
    print('Detected Colab environment. Mounting Google Drive...')
    drive.mount('/content/drive')
    # Change REPO_DIR below to the path where you placed the repo in your Drive
    REPO_DIR = '/content/drive/MyDrive/parking_spaces'
    %cd $REPO_DIR
    print('Installing dependencies from requirements.txt (may take a minute)...')
    # Install required packages (adjust path if your requirements.txt is elsewhere)
    !pip install -q -r requirements.txt
    # Ensure albumentations is available for augmentations
    !pip install -q albumentations
    print('Setup complete. Working directory:', REPO_DIR)
except Exception:
    print('Not running in Colab or automatic setup failed. If running in Colab, set REPO_DIR to your repo path and run installs manually.')

In [ ]:
### Colab setup
This cell sets up a Colab environment (Drive mount, pip installs). If running locally you can skip it. Run this cell only when starting a fresh Colab session.


In [ ]:
# 2. Configure paths and runtime settings
ROOT = Path.cwd()
SEG_JSON_GLOB = ROOT / 'data' / 'segmented_images' / '*.json'
IMG_FOLDER = ROOT / 'data' / 'raw_images'

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# Hyperparams for exploration
TARGET_SIZE = (256, 256)  # (w, h) - changed to 256 for POC
MASK_THICKNESS_PX = 3  # thickness in original-image pixels before scaling
MIN_POLY_AREA = 50  # heuristic for invalid boundaries

# Collect JSONs
json_paths = sorted(list(glob.glob(str(SEG_JSON_GLOB))))
print(f'Found {len(json_paths)} JSON label files')


In [ ]:
### Paths & runtime settings
Defines file-paths, random seeds, and dataset discovery. Edit `ROOT`, `IMG_FOLDER`, or `SEG_JSON_GLOB` if your repo layout differs.


In [ ]:
# Helper utilities

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def compute_isotropic_scale(original_w, original_h, target_w, target_h):
    s = min(target_w / original_w, target_h / original_h)
    new_w = int(round(original_w * s))
    new_h = int(round(original_h * s))
    pad_left = (target_w - new_w) // 2
    pad_top = (target_h - new_h) // 2
    return s, new_w, new_h, pad_left, pad_top


def polygon_area(px):
    # px: Nx2 array
    if px is None or len(px) < 3:
        return 0.0
    x = px[:,0]
    y = px[:,1]
    return 0.5 * np.abs(np.dot(x, np.roll(y,1)) - np.dot(y, np.roll(x,1)))


def draw_mask_for_polygon(shape, polygon, thickness=3):
    # shape: (h,w)
    mask = np.zeros(shape, dtype=np.uint8)
    if polygon is None or len(polygon) < 2:
        return mask
    pts = np.array(polygon, dtype=np.int32).reshape((-1,1,2))
    cv2.polylines(mask, [pts], isClosed=True, color=255, thickness=thickness, lineType=cv2.LINE_AA)
    return mask


def apply_transform_scale_and_pad_to_polygon(polygon, s, pad_left, pad_top):
    pts = np.array(polygon, dtype=float) * s
    pts[:,0] += pad_left
    pts[:,1] += pad_top
    return pts


In [ ]:
### Helper utilities
This cell defines helper functions used across the notebook (`load_json`, `compute_isotropic_scale`, `draw_mask_for_polygon`, etc.). Do not modify unless you know the impact on downstream cells.


In [ ]:
# 3. Load and visualize a sample image and mask

def show_sample(json_path):
    data = load_json(json_path)
    name = data.get('image_name') or Path(json_path).stem
    img_candidates = list((Path(data.get('image_name', ''))).parent.glob('*'))
    # Find image file by stem
    stem = str(Path(json_path).stem).replace('_draft','')
    candidate_imgs = list(IMG_FOLDER.glob(stem + '.*'))
    if not candidate_imgs:
        # fallback to scanning directory by stem match
        for p in IMG_FOLDER.iterdir():
            if p.stem == Path(json_path).stem:
                candidate_imgs.append(p)
    if not candidate_imgs:
        print('No image found for', json_path)
        return
    img_path = candidate_imgs[0]
    img = cv2.imread(str(img_path))  # BGR
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h0,w0 = img.shape[:2]

    poly = data.get('boundary_polygon_px')
    transform = data.get('boundary_transform', {})
    dx = transform.get('dx', 0.0)
    dy = transform.get('dy', 0.0)
    theta = transform.get('theta_deg', 0.0)

    orig_mask = draw_mask_for_polygon((h0,w0), poly, thickness=MASK_THICKNESS_PX)

    # show image, overlay
    fig, axes = plt.subplots(1,3, figsize=(14,5))
    axes[0].imshow(img)
    axes[0].set_title('Image (RGB)')
    axes[0].axis('off')

    axes[1].imshow(img)
    if poly is not None:
        pts = np.array(poly)
        axes[1].plot(pts[:,0], pts[:,1], '-r', linewidth=1)
    axes[1].set_title('Boundary overlay')
    axes[1].axis('off')

    axes[2].imshow(orig_mask, cmap='gray')
    axes[2].set_title('Binary mask (orig)')
    axes[2].axis('off')

    plt.suptitle(f'{Path(json_path).name}  dx={dx:.1f} dy={dy:.1f} theta={theta:.1f} deg')
    plt.show()

# Show up to 3 random examples
for p in random.sample(json_paths, min(3, len(json_paths))):
    show_sample(p)


In [ ]:
### Sample visualization
Runs `show_sample()` on a few labeled JSONs and displays the raw image, polygon overlay, and binary mask. Use this to sanity-check labels before training.


In [ ]:
# 4. Preprocess: resize, normalize image and binarize mask

def preprocess_image_and_mask(img_rgb, polygon, target_size=TARGET_SIZE, thickness=MASK_THICKNESS_PX, transform=None):
    h0,w0 = img_rgb.shape[:2]
    s, new_w, new_h, pad_left, pad_top = compute_isotropic_scale(w0, h0, target_size[0], target_size[1])
    # Resize image
    resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_size[1], target_size[0], 3), dtype=np.uint8)
    canvas[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized

    # Mask: draw scaled polygon on canvas directly
    mask_canvas = np.zeros((target_size[1], target_size[0]), dtype=np.uint8)
    if polygon is not None:
        scaled_pts = np.array(polygon, dtype=float) * s
        scaled_pts[:,0] += pad_left
        scaled_pts[:,1] += pad_top
        mask_canvas = draw_mask_for_polygon(mask_canvas.shape, scaled_pts, thickness=max(1, int(round(thickness * s))))

    # Apply augmentations on RGB canvas+mask before converting/normalizing
    if transform is not None:
        augmented = transform(image=canvas, mask=mask_canvas)
        canvas = augmented['image']
        mask_canvas = augmented['mask']

    # Normalize RGB canvas to [0,1]
    img_float = canvas.astype(np.float32) / 255.0
    bin_mask = (mask_canvas > 127).astype(np.uint8)
    return img_float, bin_mask, s, (new_w,new_h,pad_left,pad_top)

# Test preprocessing on a random sample
p = random.choice(json_paths)
data = load_json(p)
# find image path
stem = Path(p).stem.replace('_draft','')
img_candidates = list(IMG_FOLDER.glob(stem + '.*'))
if img_candidates:
    img_path = img_candidates[0]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    img_f, msk, s, info = preprocess_image_and_mask(img, data.get('boundary_polygon_px'))
    print('scale s=',s,' resize info=',info)
    new_w, new_h, pad_left, pad_top = info
    resized_rgb = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas_rgb = np.zeros((TARGET_SIZE[1], TARGET_SIZE[0], 3), dtype=np.uint8)
    canvas_rgb[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized_rgb
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1); plt.imshow(canvas_rgb); plt.axis('off'); plt.title('Preprocessed RGB (resized)')
    plt.subplot(1,2,2); plt.imshow(msk, cmap='gray'); plt.axis('off'); plt.title('Binary mask')
    plt.show()
else:
    print('No matching image file found for', p)

In [ ]:
### Preprocessing & resizing
This cell contains `preprocess_image_and_mask` which resizes images isotropically and draws mask lines at the configured thickness. Change `TARGET_SIZE` in the top configuration to control the downsample target.


In [ ]:
def to_combined_tensor(img_float, bin_mask):
    # img_float: HxWx3 float32 in [0,1], bin_mask: HxW uint8 (0/1)
    # Apply ImageNet normalization to RGB channels and leave mask channel in [0,1]
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_norm = (img_float - mean[None,None,:]) / std[None,None,:]
    mask_ch = bin_mask[...,None].astype(np.float32)
    combined = np.concatenate([img_norm, mask_ch], axis=2)
    # convert to CHW tensor
    t = torch.from_numpy(combined.transpose(2,0,1)).float()
    return t

# quick test
t = to_combined_tensor(img_f, msk)
print('tensor shape', t.shape, 'value range', t.min().item(), t.max().item())

In [ ]:
# 6. Synchronized data augmentation (image + mask)

# Use shared augmentation pipeline if available
try:
    from src.data.augmentations import get_common_augmentation
    transform = get_common_augmentation(target_size=TARGET_SIZE)
    if transform is None:
        print('Albumentations not available - skipping augmentations in this notebook')
    else:
        print('Using shared albumentations transform from src.data.augmentations')
except Exception as e:
    transform = None
    print('Failed to import shared augmentation function (albumentations may be missing):', e)

# Demo augmentation (apply to RGB canvas before conversion/normalization)
if transform is not None:
    # Build RGB canvas for demo
    p = random.choice(json_paths)
    data = load_json(p)
    stem = Path(p).stem.replace('_draft','')
    img_candidates = list(IMG_FOLDER.glob(stem + '.*'))
    if img_candidates:
        img_path = img_candidates[0]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img_f, msk, s, info = preprocess_image_and_mask(img, data.get('boundary_polygon_px'))
        new_w, new_h, pad_left, pad_top = info
        resized_rgb = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas_rgb = np.zeros((TARGET_SIZE[1], TARGET_SIZE[0], 3), dtype=np.uint8)
        canvas_rgb[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized_rgb
        aug = transform(image=canvas_rgb, mask=msk)
        img_aug = aug['image'].astype(np.float32) / 255.0
        mask_aug = aug['mask']
        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1); plt.imshow(img_aug); plt.axis('off'); plt.title('Augmented image')
        plt.subplot(1,2,2); plt.imshow(mask_aug, cmap='gray'); plt.axis('off'); plt.title('Augmented mask')
        plt.show()

In [ ]:
# 7. PyTorch Dataset and DataLoader returning combined tensors

class BoundaryDataset(Dataset):
    def __init__(self, json_paths, img_folder, target_size=(256,256), thickness=3, transform=None, min_area=MIN_POLY_AREA):
        self.items = json_paths
        self.img_folder = Path(img_folder)
        self.target_size = target_size
        self.thickness = thickness
        self.transform = transform
        self.min_area = min_area

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        jp = self.items[idx]
        data = load_json(jp)
        stem = Path(jp).stem.replace('_draft','')
        img_candidates = list(self.img_folder.glob(stem + '.*'))
        if not img_candidates:
            raise FileNotFoundError(f'No image for {jp}')
        img = cv2.cvtColor(cv2.imread(str(img_candidates[0])), cv2.COLOR_BGR2RGB)
        poly = data.get('boundary_polygon_px')
        transform_meta = data.get('boundary_transform', {})
        dx = float(transform_meta.get('dx', 0.0))
        dy = float(transform_meta.get('dy', 0.0))
        theta_deg = float(transform_meta.get('theta_deg', 0.0))

        # Preprocess: resize and build RGB canvas
        h0, w0 = img.shape[:2]
        s, new_w, new_h, pad_left, pad_top = compute_isotropic_scale(w0, h0, self.target_size[0], self.target_size[1])
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((self.target_size[1], self.target_size[0], 3), dtype=np.uint8)
        canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized

        mask_canvas = np.zeros((self.target_size[1], self.target_size[0]), dtype=np.uint8)
        if poly is not None:
            scaled_pts = (np.array(poly, dtype=float) * s)
            scaled_pts[:, 0] += pad_left
            scaled_pts[:, 1] += pad_top
            t_thick = max(1, int(round(self.thickness * s)))
            mask_canvas = draw_mask_for_polygon(mask_canvas.shape, scaled_pts, thickness=t_thick)

        # Apply augmentation if provided (on RGB canvas)
        if self.transform is not None:
            augmented = self.transform(image=canvas, mask=mask_canvas)
            canvas = augmented['image']
            mask_canvas = augmented['mask']

        # Normalize RGB canvas to [0,1]
        img_f = canvas.astype(np.float32) / 255.0
        bin_mask = (mask_canvas > 127).astype(np.uint8)

        info = (new_w, new_h, pad_left, pad_top)

        valid = 1
        if poly is None or polygon_area(np.array(poly)) < self.min_area:
            valid = 0
        # scale dx/dy
        scaled_dx = s * dx
        scaled_dy = s * dy
        dx_norm = scaled_dx / (self.target_size[0] / 2.0)
        dy_norm = scaled_dy / (self.target_size[1] / 2.0)
        theta_rad = math.radians(theta_deg)
        cos_t = math.cos(theta_rad)
        sin_t = math.sin(theta_rad)

        combined_t = to_combined_tensor(img_f, bin_mask)

        target = {
            'valid': torch.tensor(valid, dtype=torch.float32),
            'dx_norm': torch.tensor(dx_norm, dtype=torch.float32),
            'dy_norm': torch.tensor(dy_norm, dtype=torch.float32),
            'cos': torch.tensor(cos_t, dtype=torch.float32),
            'sin': torch.tensor(sin_t, dtype=torch.float32),
            'scale_s': s,
            'json_path': jp
        }
        return combined_t, target

# instantiate a small dataloader
sample_dataset = BoundaryDataset(json_paths, IMG_FOLDER, target_size=TARGET_SIZE, thickness=MASK_THICKNESS_PX, transform=transform)
loader = DataLoader(sample_dataset, batch_size=4, shuffle=True)
print('Dataset size:', len(sample_dataset))

# show a minibatch
for batch in loader:
    imgs, targets = batch
    print('imgs', imgs.shape)
    print('target_valid', [t['valid'].item() for t in targets])
    break

In [ ]:
# Checkpointing & live visualization utilities

import os
import time
import csv
from IPython.display import clear_output

# Save directory (set to Drive path in Colab; otherwise local notebooks/outputs)
save_dir = Path(os.environ.get('PARKING_SAVE_DIR', './experiments'))
save_dir.mkdir(parents=True, exist_ok=True)
print('Saving outputs to', save_dir)

# Checkpoint saver (atomic)
def save_checkpoint(epoch, model, opt, best_val, save_dir, prefix='epoch'):
    ckpt = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'opt_state': opt.state_dict(),
        'best_val': best_val,
        'timestamp': time.time()
    }
    tmp = str(save_dir / f'{prefix}_{epoch}.pth.tmp')
    final = str(save_dir / f'{prefix}_{epoch}.pth')
    torch.save(ckpt, tmp)
    os.replace(tmp, final)
    return final

# Append metrics to CSV
metrics_csv = save_dir / 'metrics.csv'
if not metrics_csv.exists():
    with open(metrics_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss', 'timestamp'])

def append_metrics(epoch, train_loss, val_loss, save_dir):
    with open(save_dir / 'metrics.csv', 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f'{train_loss:.6f}', f'{val_loss:.6f}', time.time()])

# Live plot helper
from matplotlib import pyplot as plt

def plot_losses(train_losses, val_losses):
    clear_output(wait=True)
    plt.figure(figsize=(6,4))
    plt.plot(range(1, len(train_losses)+1), train_losses, '-o', label='train')
    plt.plot(range(1, len(val_losses)+1), val_losses, '-o', label='val')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Loss curves')
    plt.show()

# Save sample visualizations
def save_epoch_visualizations(model, val_dataset, save_dir, epoch, n_samples=3, device='cpu'):
    model.eval()
    with torch.no_grad():
        for jp in val_dataset[:n_samples]:
            # val_dataset expects list of json paths; create a small dataset instance
            ds = BoundaryDataset([jp], IMG_FOLDER, target_size=TARGET_SIZE, thickness=MASK_THICKNESS_PX, transform=None)
            x, t = ds[0]
            x_b = x.unsqueeze(0).to(device)
            valid_logits, reg = model(x_b)
            valid_p = torch.sigmoid(valid_logits).item()
            reg_np = reg.cpu().numpy()[0]
            out = apply_predicted_transform_to_original_polygon(jp, reg_np, target_size=TARGET_SIZE)
            if out is None:
                continue
            img_orig, poly_orig, poly_pred = out
            fig, ax = plt.subplots(1,1, figsize=(6,6))
            ax.imshow(img_orig)
            pts = np.array(poly_orig)
            ax.plot(pts[:,0], pts[:,1], '-r', linewidth=2, label='original (red)')
            p2 = np.array(poly_pred)
            ax.plot(p2[:,0], p2[:,1], '-y', linewidth=2, label='predicted (yellow)')
            ax.axis('off')
            ax.legend()
            fn = save_dir / f'viz_epoch_{epoch}_{Path(jp).stem}.png'
            fig.savefig(fn, bbox_inches='tight')
            plt.close(fig)

print('Checkpoint & visualization utilities ready')

In [ ]:
# Training demo with checkpointing and live visualization

# Note: This is a small proof-of-concept training loop intended for Colab. For robust evaluation
# on tiny datasets prefer k-fold cross-validation or repeated random splits.

# simple train/val split (deterministic)
paths = json_paths.copy()
random.Random(RANDOM_SEED).shuffle(paths)
train_paths = paths[:max(1, len(paths) - 6)]  # leave up to 6 for val
val_paths = paths[len(train_paths):]
print(f'Train: {len(train_paths)}, Val: {len(val_paths)}')

train_ds = BoundaryDataset(train_paths, IMG_FOLDER, target_size=TARGET_SIZE, thickness=MASK_THICKNESS_PX, transform=transform)
val_ds = BoundaryDataset(val_paths, IMG_FOLDER, target_size=TARGET_SIZE, thickness=MASK_THICKNESS_PX, transform=None)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

# model
model = BoundaryRegressor(pretrained=True, in_channels=4).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# training settings with early stopping
num_epochs = 12
train_losses = []
val_losses = []
best_val = float('inf')
patience = 3
min_delta = 1e-4
epochs_no_improve = 0
save_every_n = 1
saved_best = None

for epoch in range(1, num_epochs + 1):
    model.train()
    running = 0.0
    count = 0
    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        opt.zero_grad()
        valid_logits, reg = model(imgs)
        loss, metrics = loss_fn(valid_logits, reg, targets)
        loss.backward()
        opt.step()
        running += loss.item()
        count += 1
    epoch_train_loss = running / max(1, count)
    train_losses.append(epoch_train_loss)

    # validation
    model.eval()
    with torch.no_grad():
        vrunning = 0.0
        vcount = 0
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            valid_logits, reg = model(imgs)
            loss, _m = loss_fn(valid_logits, reg, targets)
            vrunning += loss.item()
            vcount += 1
        epoch_val_loss = vrunning / max(1, vcount)
        val_losses.append(epoch_val_loss)

    print(f'Epoch {epoch}/{num_epochs} - train_loss={epoch_train_loss:.4f} val_loss={epoch_val_loss:.4f}')

    # persist metrics and checkpoints
    append_metrics(epoch, epoch_train_loss, epoch_val_loss, save_dir)
    if epoch % save_every_n == 0:
        print('Saving epoch checkpoint...')
        save_checkpoint(epoch, model, opt, best_val, save_dir, prefix='epoch')

    # early stopping logic
    if epoch_val_loss < best_val - min_delta:
        best_val = epoch_val_loss
        saved_best = save_checkpoint(epoch, model, opt, best_val, save_dir, prefix='best')
        epochs_no_improve = 0
        print('New best model saved to', saved_best)
    else:
        epochs_no_improve += 1
        print(f'No improvement for {epochs_no_improve}/{patience} epochs')
        if epochs_no_improve >= patience:
            print('Early stopping triggered - stopping training')
            break

    # save validation visualizations
    try:
        save_epoch_visualizations(model, val_paths, save_dir, epoch, n_samples=min(3, len(val_paths)), device=device)
    except Exception as e:
        print('Warning saving visuals:', e)

    # live plot
    try:
        plot_losses(train_losses, val_losses)
    except Exception as e:
        print('Warning plotting losses:', e)

# Load best checkpoint if available
if saved_best is not None:
    print('Loading best checkpoint', saved_best)
    ckpt = torch.load(saved_best, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    print('Best checkpoint loaded')

# final save
save_checkpoint('final', model, opt, best_val, save_dir, prefix='final')
print('Training complete - final checkpoint saved')

In [ ]:
# Inference: interactive image selector + detect red boundary, predict transform, and overlay predicted boundary (yellow) + detected red boundary

from app.core.boundary import estimate_boundary_from_overlay
from app.core.render import draw_polygon

# Build list of available labeled JSONs
available_jps = val_paths if 'val_paths' in globals() and len(val_paths) > 0 else json_paths
print(f'Found {len(available_jps)} labeled JSONs available for inference')

# Interactive selection (ipywidgets if available, otherwise fallback to console input)
selected_jp = None
try:
    import ipywidgets as widgets
    options = [(str(Path(p).name), str(p)) for p in available_jps]
    dropdown = widgets.Dropdown(options=options, description='Image')
    select_btn = widgets.Button(description='Select')
    out = widgets.Output()

    def on_select(b):
        nonlocal selected_jp
        selected_jp = dropdown.value
        with out:
            print('Selected:', selected_jp)
    select_btn.on_click(on_select)
    display(dropdown, select_btn, out)
    print('Click Select to choose an image. Or set `selected_jp` variable manually and re-run this cell.')
except Exception:
    # Console fallback
    print('ipywidgets not available - falling back to console selection')
    for i, p in enumerate(available_jps):
        print(f'{i}: {Path(p).name}')
    idx = int(input('Enter index of image to run inference on: '))
    selected_jp = available_jps[idx]

# Wait until selected_jp is set (works in interactive sessions)
if selected_jp is None:
    print('No selection made yet. Set `selected_jp` and re-run the inference cell.')
else:
    print('Running inference for', selected_jp)

    # Helper: apply predicted reg (dx_norm, dy_norm, cos, sin) to an arbitrary polygon in ORIGINAL image pixels
    def apply_pred_to_polygon(orig_poly, reg_arr, orig_w, orig_h, target_size=TARGET_SIZE):
        dx_norm, dy_norm, cos_p, sin_p = reg_arr
        dx_px_scaled = dx_norm * (target_size[0] / 2.0)
        dy_px_scaled = dy_norm * (target_size[1] / 2.0)
        theta_rad = math.atan2(sin_p, cos_p)
        s, new_w, new_h, pad_left, pad_top = compute_isotropic_scale(orig_w, orig_h, target_size[0], target_size[1])
        if s == 0:
            return None
        dx_orig = float(dx_px_scaled / s)
        dy_orig = float(dy_px_scaled / s)
        orig_poly = np.array(orig_poly, dtype=float)
        centroid = orig_poly.mean(axis=0)
        cos_t = math.cos(theta_rad)
        sin_t = math.sin(theta_rad)
        R = np.array([[cos_t, -sin_t], [sin_t, cos_t]])
        b = orig_poly - centroid
        b = b @ R.T
        b = b + centroid + np.array([dx_orig, dy_orig])
        return b

    # Single-case inference
    jp = selected_jp
    stem = Path(jp).stem.replace('_draft', '')
    img_files = list(IMG_FOLDER.glob(stem + '.*'))
    if not img_files:
        print('No image file for', jp)
    else:
        img_path = img_files[0]
        bgr = cv2.imread(str(img_path))  # BGR
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h0, w0 = rgb.shape[:2]

        boundary_result = estimate_boundary_from_overlay(bgr)
        if boundary_result.polygon_px is None:
            print(Path(jp).name, '— no boundary detected (confidence', boundary_result.confidence, ')')
        else:
            detected_poly = boundary_result.polygon_px
            img_f, mask_f, s, info = preprocess_image_and_mask(rgb, detected_poly, target_size=TARGET_SIZE, thickness=MASK_THICKNESS_PX)
            inp = to_combined_tensor(img_f, mask_f).unsqueeze(0).to(device)

            with torch.no_grad():
                valid_logits, reg = model(inp)
                valid_p = float(torch.sigmoid(valid_logits).cpu().numpy())
                reg_np = reg.cpu().numpy()[0]

            poly_pred = apply_pred_to_polygon(detected_poly, reg_np, orig_w=w0, orig_h=h0, target_size=TARGET_SIZE)

            vis = bgr.copy()
            vis = draw_polygon(vis, detected_poly, color=(0, 0, 255), thickness=2)
            if poly_pred is not None:
                vis = draw_polygon(vis, poly_pred.tolist(), color=(0, 255, 255), thickness=2)

            vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(8,8))
            plt.imshow(vis_rgb)
            plt.title(f"{Path(jp).name} - valid_prob={valid_p:.2f}")
            plt.axis('off')
            plt.show()

            # Save visualization to save_dir
            out_fn = save_dir / f'overlay_{Path(jp).stem}.png'
            cv2.imwrite(str(out_fn), vis)
            print('Saved overlay to', out_fn)